## Downloading Footbal Domain Data

### Helper websites

https://www.w3.org/TR/rdf-sparql-query/#describe <br>
https://sparqlwrapper.readthedocs.io/en/latest/main.html#return-formats <br>
https://stackoverflow.com/questions/40528578/how-can-i-properly-serialize-wikidata-sparql-queries-answers <br>
https://wiki.app.uib.no/info216/index.php/Python_Examples#Retrieving_data_from_Wikidata_with_SparqlWrapper <br>


In [45]:
from SPARQLWrapper import SPARQLWrapper, JSON
from rdflib import Graph
import datetime
import pandas as pd

In [31]:
sparql = SPARQLWrapper(endpoint='https://query.wikidata.org/')
sparql.setQuery("""
PREFIX pq: <http://www.wikidata.org/prop/qualifier/>
PREFIX ps: <http://www.wikidata.org/prop/statement/>
PREFIX p: <http://www.wikidata.org/prop/>
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>

CONSTRUCT {?player p:P54 ?x}
WHERE {?player wdt:P106 wd:Q937857}

LIMIT 20

""")


sparql.setReturnFormat(JSON)
results = sparql.query().convert()

# the destination file, with code to make it timestamped
#destfile = 'rdf_dumps/test-' + datetime.datetime.now().strftime('%Y%m%d-%H%M') + '.rdf'
print(results)
# serialize the result to file
#results.serialize(destination=destfile, format='ttl')


b'<!DOCTYPE html><html lang="en" dir="ltr"><head><meta charset="utf-8"><meta http-equiv="X-UA-Compatible" content="IE=edge"><meta name="viewport" content="width=device-width,initial-scale=1,user-scalable=yes"><link rel="stylesheet" href="css/style.min.372a7e7a49dcaedce6ae.css"><link id="favicon" rel="shortcut icon"><script src="js/shim.min.05fe49cdd30866fd4947.js"></script><style id="MJX-CHTML-styles">/* placeholder for MathJax */</style></head><body><div class="wikibase-queryservice container-fluid"><div class="row"><nav class="navbar navbar-default"><div class="navbar-header"><button type="button" class="navbar-toggle collapsed" data-toggle="collapse" data-target="#header-navbar-collapse" aria-expanded="false"><span class="sr-only">Toggle navigation</span><span class="icon-bar"></span><span class="icon-bar"></span><span class="icon-bar"></span></button><div class="navbar-brand"><a href="./"><span></span></a></div></div><div class="collapse navbar-collapse in" id="header-navbar-collap

In [15]:
from SPARQLWrapper import SPARQLWrapper, XML
from rdflib import Graph

sparql = SPARQLWrapper("http://dbpedia.org/sparql")
sparql.setQuery("""
    PREFIX dbo: <http://dbpedia.org/ontology/>
    PREFIX schema: <http://schema.org/>
    CONSTRUCT {
      ?lang a schema:Language ;
      schema:alternateName ?iso6391Code . 
    }
    WHERE {
      ?lang a dbo:Language ;
      dbo:iso6391Code ?iso6391Code .
      FILTER (STRLEN(?iso6391Code)=2) # to filter out non-valid values
    }
""")


results = sparql.query().convert()
print(results.serialize())

@prefix schema: <http://schema.org/> .

<http://dbpedia.org/resource/Abkhaz_language> a schema:Language ;
    schema:alternateName "ab" .

<http://dbpedia.org/resource/Afar_language> a schema:Language ;
    schema:alternateName "aa" .

<http://dbpedia.org/resource/Afrikaans> a schema:Language ;
    schema:alternateName "af" .

<http://dbpedia.org/resource/Akan_language> a schema:Language ;
    schema:alternateName "ak" .

<http://dbpedia.org/resource/Albanian_language> a schema:Language ;
    schema:alternateName "sq" .

<http://dbpedia.org/resource/Amharic> a schema:Language ;
    schema:alternateName "am" .

<http://dbpedia.org/resource/Arabic> a schema:Language ;
    schema:alternateName "ar" .

<http://dbpedia.org/resource/Aragonese_language> a schema:Language ;
    schema:alternateName "an" .

<http://dbpedia.org/resource/Armenian_language> a schema:Language ;
    schema:alternateName "hy" .

<http://dbpedia.org/resource/Assamese_language> a schema:Language ;
    schema:alternateN

In [51]:
# Gives you player played_for team details
sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setReturnFormat(JSON)

played_for_query = """


PREFIX pq: <http://www.wikidata.org/prop/qualifier/>
PREFIX ps: <http://www.wikidata.org/prop/statement/>
PREFIX p: <http://www.wikidata.org/prop/>
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>

SELECT ?player ?relation ?team
WHERE {
   ?player wdt:P106 wd:Q937857.

   ?player p:P54 ?position_held_statement .
   ?position_held_statement ps:P54 ?team .
  VALUES ?relation {p:P54}.
  }
limit 20
"""

sparql.setQuery(played_for_query)


played_for_results = sparql.query().convert()

for result in played_for_results["results"]["bindings"]:
    #pass
    print("Player"," ","played_for", "Team")
    print(result["player"]["value"], "   ", result["relation"]["value"], "   ",result["team"]["value"])
    break;

Player   played_for Team
http://www.wikidata.org/entity/Q10723     http://www.wikidata.org/prop/P54     http://www.wikidata.org/entity/Q2714


In [60]:
#pull temporal data for player and when he played for a team

played_for_time_query ="""

SELECT ?player ?team ?start ?end
WHERE {
   ?player wdt:P106 wd:Q937857.

   ?player p:P54 ?position_held_statement .
   ?position_held_statement ps:P54 ?team .
   ?position_held_statement pq:P580 ?start .
  ?position_held_statement pq:P582 ?end .
  
 } 
limit 20

"""
sparql.setQuery(played_for_time_query)


played_for_time_results = sparql.query().convert()

for result in played_for_time_results["results"]["bindings"]:
    #pass
    print("Player"," ","played_for", "Start","end")
    print(result["player"]["value"], "   ", result["team"]["value"], "   ",result["start"]["value"]," ",result["end"]["value"])
    break;


Player   played_for Start end
http://www.wikidata.org/entity/Q19365     http://www.wikidata.org/entity/Q10333     2008-01-01T00:00:00Z   2014-01-01T00:00:00Z


In [52]:
# gives you team and its country 
#Note: some teams have multiple country values based on time

teams_country_query = """

SELECT ?team ?relation ?country
WHERE {
   ?player wdt:P106 wd:Q937857.

   ?player p:P54 ?team_played_statement .
   ?team_played_statement ps:P54 ?team .
   ?team p:P17 ?country_statement.
   ?country_statement ps:P17 ?country.
  VALUES ?relation {p:P17}.
  }
limit 20
"""
sparql.setQuery(teams_country_query)

teams_country_results = sparql.query().convert()

for result in teams_country_results["results"]["bindings"]:
    #pass
    print("Team"," ","played_for", "Team")
    print(result["team"]["value"], "   ", result["relation"]["value"], "   ",result["country"]["value"])
    break;

Team   played_for Team
http://www.wikidata.org/entity/Q54834034     http://www.wikidata.org/prop/P17     http://www.wikidata.org/entity/Q159


In [54]:
#Pull temporal data for team and its country here

teams_country_time_query ="""

SELECT ?team ?country 
WHERE {
   ?player wdt:P106 wd:Q937857.

   ?player p:P54 ?team_played_statement .
   ?team_played_statement ps:P54 ?team .
   ?team p:P17 ?country_statement.
   ?country_statement ps:P17 ?country.
  VALUES ?relation {p:P17}.
  }
limit 20

"""
sparql.setQuery(played_for_time_query)


played_for_time_results = sparql.query().convert()

for result in played_for_time_results["results"]["bindings"]:
    #pass
    print("Player"," ","played_for", "Start","end")
    print(result["player"]["value"], "   ", result["team"]["value"], "   ",result["start"]["value"]," ",result["end"]["value"])
    break;



In [53]:
# gives you team and the league
#Note: some teams have multiple country values based on time

teams_league_query = """

SELECT ?team ?relation ?league
WHERE {
   ?player wdt:P106 wd:Q937857. # player belonging to football association

   ?player p:P54 ?team_played_statement .
   ?team_played_statement ps:P54 ?team .
   ?team p:P118 ?league_statement.
   ?league_statement ps:P118 ?league.
  VALUES ?relation {p:P118}.
  }
limit 20
"""
sparql.setQuery(teams_league_query)

teams_league_results = sparql.query().convert()

for result in teams_league_results["results"]["bindings"]:
    #pass
    print("Team"," ","is in", "League")
    print(result["team"]["value"], "   ", result["relation"]["value"], "   ",result["league"]["value"])
    break;

Team   is in League
http://www.wikidata.org/entity/Q19481     http://www.wikidata.org/prop/P118     http://www.wikidata.org/entity/Q9448
